In [1]:
# import libraries

import pymupdf
import spacy
import re
import pandas as pd
import numpy as np
#import unicodedata
import os
from pathlib import Path
from translate import Translator

In [4]:
nlp_en_lg = spacy.load('en_core_web_lg')
nlp_en_md = spacy.load('en_core_web_md')
nlp_en_sm = spacy.load('en_core_web_sm')

In [5]:
# compare noun chunk approach to UOP words similarity 
# /Users/elisabethwatson/Documents/GitHub/ADSCert_project_Watson/PDF1/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf

text = "The proceeds will be fully allocated to (re) finance 49 distributed solar energy generation projects. ' \
'The financed projects meet the eligibility requirements in the Climate Bonds Taxonomy and are aligned with the GBP. ' \
'In the portfolio of projects to be financed, there are nine plants already under installation, as table below"

In [7]:
doc = nlp_en_sm(text)
for chunk in doc.noun_chunks:
    print(chunk.text, chunk.root.text, chunk.root.dep_, chunk.root.head.text)

The proceeds proceeds nsubjpass allocated
re re pobj to
finance finance pobj to
49 distributed solar energy generation projects projects pobj to
The financed projects projects nsubj meet
the eligibility requirements requirements dobj meet
the Climate Bonds Taxonomy Taxonomy pobj in
the GBP GBP pobj with
the portfolio portfolio pobj In
projects projects pobj of
nine plants plants attr are
installation installation pobj under
table table pobj as


In [8]:
# create lingusitic features for ICMA doc to learn the syntactic patterns
# /Users/elisabethwatson/Documents/GitHub/ADSCert_project_Watson/Green-Bond-Principles-GBP-June-2025.pdf

# to find the Use of Proceeds portion of the document, returns font info for key phrases that typically indicate the start and end of that portion

def fontInfo(language, document):

    results_UOP = []
    results_SEEGP = []
    
    pdf = pymupdf.open(document)
    
    if language == 'PT':
        keywordsUOP = ['Usos dos Recursos', 'Uso de Recursos', 'Uso dos Recursos']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']
        keywordsSEEGP_test = ['Avaliação e Seleção de Ativos', 'Processo de Avaliação e Seleção de Projetos']
    elif language == 'ES':
        keywordsUOP = ['Uso de los Fondos', 'Uso de Fondos', 'Uso de Recursos', 'Destinos de Fondos', 'Uso de los Recursos']
        keywordsSEEGP_test = ['Proceso de Evaluación y Selección de Proyectos', 'Selección y Evaluación de Proyectos', 'Procedimiento Selección', 'Evaluación y selección de proyectos']
    elif language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']
        keywordsSEEGP_test = ['Selection and Evaluation', 'Process for', 'Evaluation and Selection', 'Project Evaluation', 'Assessment Process', 'Selection process', 'Project selection criteria']


    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]
        dict = page.get_text("dict")
        blocks = dict["blocks"] 
        for block in blocks:
            if "lines" in block.keys():
                spans = block['lines']
                for span in spans:
                    data = span['spans']
                    for lines in data:
                        for keyword in keywordsUOP:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_UOP.append((lines['text'], lines['size'], lines['bbox'], page_idx))
                        for keyword in keywordsSEEGP_test:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_SEEGP.append((lines['text'], lines['size'], lines['bbox'], page_idx))

                            

    return results_UOP, results_SEEGP

# from the results of the previous function, returns the result index containing the highest font size, assuming this is a section header 

def find_max_font(results_UOP, results_SEEGP):
    
    max_font_size_UOP = 0
    if len(results_UOP) == 0:
        max_idx_UOP = 0
    else:
        for result_idx_UOP in range(len(results_UOP)):
            resultA = results_UOP[result_idx_UOP]
            if resultA[1] > max_font_size_UOP:
                max_font_size_UOP = resultA[1]
                max_idx_UOP = result_idx_UOP

    max_font_size_SEEGP = 0
    if len(results_SEEGP) == 0:
        max_idx_SEEGP = 0
    else:       
        for result_idx_SEEGP in range(len(results_SEEGP)):
            resultB = results_SEEGP[result_idx_SEEGP]
            if resultB[1] > max_font_size_SEEGP:
                max_font_size_SEEGP = resultB[1]
                max_idx_SEEGP = result_idx_SEEGP

    return max_idx_UOP, max_idx_SEEGP

In [16]:
icma_gbp = '/Users/elisabethwatson/Documents/GitHub/ADSCert_project_Watson/Green-Bond-Principles-GBP-June-2025.pdf'
language = 'EN'

results_UOP, results_SEEGP = fontInfo(language, icma_gbp)
max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)
print(max_idx_UOP, max_idx_SEEGP)
print(f"{results_UOP[max_idx_UOP]} \n {results_SEEGP[max_idx_SEEGP]}")


3 3
('1. Use of Proceeds', 12.0, (306.1416931152344, 109.67951965332031, 411.3204650878906, 123.7555160522461), 3) 
 ('2. Process for Project Evaluation and Selection ', 12.0, (36.0, 109.67951965332031, 292.4713134765625, 123.7555160522461), 5)
